In [1]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
import os 
from tabulate import tabulate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report 
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv(r'C:\Irrigation_Water_Requirement\Data\Raw_Data\irrigation_prediction.csv')
df.head()


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Rainfed,Reservoir,4.73,Yes,1.98,South,Low
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Canal,Groundwater,12.22,Yes,33.56,Central,Medium
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Drip,Reservoir,5.52,Yes,34.62,South,Low
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Canal,Reservoir,1.43,Yes,84.03,North,Medium
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,Canal,River,2.52,No,60.86,South,Medium


# EDA

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Soil_Type                10000 non-null  object 
 1   Soil_pH                  9999 non-null   float64
 2   Soil_Moisture            9994 non-null   float64
 3   Organic_Carbon           9998 non-null   float64
 4   Electrical_Conductivity  10000 non-null  float64
 5   Temperature_C            9992 non-null   float64
 6   Humidity                 9998 non-null   float64
 7   Rainfall_mm              9994 non-null   float64
 8   Sunlight_Hours           9997 non-null   float64
 9   Wind_Speed_kmh           9995 non-null   object 
 10  Crop_Type                9997 non-null   object 
 11  Crop_Growth_Stage        9996 non-null   object 
 12  Season                   9999 non-null   object 
 13  Irrigation_Type          9998 non-null   object 
 14  Water_Source           

We still have missing data even though they are not a lot.

In [4]:
df.nunique()

Soil_Type                     4
Soil_pH                     341
Soil_Moisture              4750
Organic_Carbon              131
Electrical_Conductivity     341
Temperature_C              2895
Humidity                   5305
Rainfall_mm                9808
Sunlight_Hours              702
Wind_Speed_kmh             1937
Crop_Type                     7
Crop_Growth_Stage             5
Season                        4
Irrigation_Type               5
Water_Source                  5
Field_Area_hectare         1470
Mulching_Used                 3
Previous_Irrigation_mm     6843
Region                        5
Irrigation_Need               3
dtype: int64

Target values contain 3 different categories of water requirement. 

In [5]:
df.Irrigation_Need.value_counts()

Irrigation_Need
Low       5864
Medium    3800
High       336
Name: count, dtype: int64

# Model Building

STEPS of Baseline Model: 

    ~ Splitting Dataset into Train and Test sets to overcome Data Leakage
    ~ Changing target values into numeric values (Low -> 0, Medium -> 1, High-> 2)
    ~ Handling missing values (train and test separately)
    ~ Encoding train set and test separately
    ~ Scaling train set and test sets
    ~ Model Training
    ~ Comapring model Accuarcy, Precision, Recall_Score, F1_score

Extracting Target value

In [6]:
X = df.drop(columns='Irrigation_Need')
y = df['Irrigation_Need']

Splitting Dataset into Train and Test sets

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

Checking shapes of sets

In [8]:
X_train.shape

(8000, 19)

In [9]:
X_test.shape

(2000, 19)

In [10]:
y_train.shape

(8000,)

In [11]:
y_test.shape

(2000,)

Changing types of values to numeric

In [12]:
y_train = y_train.replace({'Low': int(0),
                           'Medium':int(1),
                           'High':int(2)})

y_test = y_test.replace({'Low': int(0),
                           'Medium':int(1),
                           'High':int(2)})

C:\Users\jyusu\AppData\Local\Temp\ipykernel_3860\2869880127.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({'Low': int(0),
C:\Users\jyusu\AppData\Local\Temp\ipykernel_3860\2869880127.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({'Low': int(0),


In [13]:
y_train

9254    0
1561    0
1670    0
6087    1
6669    0
       ..
5734    0
5191    1
5390    0
860     0
7270    0
Name: Irrigation_Need, Length: 8000, dtype: int64

In [14]:
y_test

6252    0
4684    1
1731    2
4742    0
4521    0
       ..
6412    1
8285    0
7853    1
1095    0
6929    1
Name: Irrigation_Need, Length: 2000, dtype: int64

Checked to make sure that our sets are in good shape

# Preprocessing

    ~ Handling missing values

In [15]:
class missing_values:

    def __init__(self, X_train, X_test):
        self.X_train = X_train
        self.X_test = X_test
    
    def filling(self):

        for col in self.X_train.columns:
            if self.X_train[col].dtype == 'object':
                fill_values = self.X_train[col].mode()[0]
            else:
                fill_values = self.X_train[col].mean()
            
            self.X_train[col] = self.X_train[col].fillna(fill_values)
            self.X_test[col] = self.X_test[col].fillna(fill_values)
        return self


    ~ Encoding of train set and test sets

In [16]:
class encoding:

    def __init__(self, X_train, X_test, threshold = 2):
        
        self.X_train = X_train
        self.X_test = X_test
        self.threshold = threshold
        self.encoders = {}
    
    def train_encode(self):

        for col in self.X_train.columns:
            if self.X_train[col].dtype == 'object':
                if self.X_train[col].nunique() <= self.threshold:
                    dummies = pd.get_dummies(self.X_train[col], prefix=col, dtype = int)
                    self.X_train = pd.concat([self.X_train.drop(columns = col), dummies], axis=1)
                else:
                    label_enc = LabelEncoder()
                    self.X_train[col] = label_enc.fit_transform(self.X_train[col])
                    self.encoders[col] = label_enc
        return self
    
    def test_encode(self):

        for col, encoder in self.encoders.items():
            self.X_test[col] = self.X_test[col].astype(str)
            encoded = []
            for val in self.X_test[col]:
                if val in encoder.classes_:
                    encoded.append(encoder.transform([val])[0])
                else:
                    encoded.append(-1)
                    
            self.X_test[col] = encoded
        
        self.X_test = pd.get_dummies(self.X_test, dtype=int)
        self.X_test = self.X_test.reindex(columns= self.X_train.columns, fill_value=0)

        return self

    ~Scaling train set and test sets

In [17]:
class scaling:

    def __init__(self, X_train, X_test):
        self.X_train = X_train
        self.X_test = X_test
        self.scalars = {}    

    def train_scaling(self):
        for col in self.X_train.columns:
            if self.X_train[col].dtype !='object':
                scalar = MinMaxScaler()
                self.X_train[col] = scalar.fit_transform(self.X_train[[col]])
                self.scalars[col] = scalar

        return self
    
    def test_scaling(self):

        for col in self.X_test.columns:
            if col in self.scalars:
                self.X_test[col] = self.scalars[col].transform(self.X_test[[col]])

        return self

    ~Using Classes to do preprocessing

In [18]:
mis = missing_values(X_train = X_train, X_test = X_test)
mis.filling()

Saving this filled data to Preporcessed folder

In [19]:
folder = "C:/Irrigation_Water_Requirement/Data/Preprocesssed_Data/missed_data"

path = os.path.join(folder, 'X_train_filled.csv')
X_train.to_csv(path, index=False)

In [20]:
path = os.path.join(folder, 'X_test_filled.csv')
X_test.to_csv(path, index=False)

Running encoding class

In [21]:
enc = encoding(X_train = X_train, X_test = X_test, threshold=2)
enc.train_encode()

In [22]:
enc.test_encode()

Saving encoded set

In [23]:
folder = "C:/Irrigation_Water_Requirement/Data/Preprocesssed_Data/encoded_data"

path = os.path.join(folder, 'X_train_encoder.csv')
X_train.to_csv(path, index=False)


In [24]:
path = os.path.join(folder, 'X_test_encoder.csv')
X_test.to_csv(path, index=False)

In [25]:
X_test

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
6252,0,7.85,15.25,1.18,1.41,25.32,42.38,602.97,9.88,1908,5,3,3,1,1,711,2,1633,3
4684,1,6.68,13.47,1.05,2.80,17.21,45.31,1922.03,4.71,1034,4,0,1,2,1,1007,2,2203,3
1731,0,5.56,11.00,0.91,0.40,26.62,70.89,11.02,7.94,499,6,0,2,4,3,170,1,-1,3
4742,3,7.70,38.58,1.19,1.54,18.71,74.54,1169.81,5.03,57,4,1,2,2,1,882,1,-1,2
4521,0,6.88,47.06,1.16,2.44,14.90,68.39,312.59,7.49,254,3,1,3,4,1,1002,1,2514,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6412,0,5.05,49.77,0.81,2.99,21.80,54.45,2225.80,5.11,994,4,4,1,1,1,1454,1,3959,0
8285,2,8.05,41.09,1.43,1.55,17.51,73.15,704.61,4.25,72,2,4,3,2,2,1396,2,-1,0
7853,3,6.59,42.69,1.53,2.62,16.00,63.75,292.44,6.64,907,5,4,1,4,4,1085,1,5823,4
1095,1,7.32,63.85,1.18,3.35,26.52,81.91,1104.52,6.01,1101,4,1,2,1,2,571,1,-1,3


Running scaling class

In [26]:
scale = scaling(X_train, X_test)
scale.train_scaling()
scale.test_scaling()

Saving scaled set

In [27]:
folder = "C:/Irrigation_Water_Requirement/Data/Preprocesssed_Data/scaled_data"
path = os.path.join(folder, 'X_train_scaled.csv')
X_train.to_csv(path, index=False)

In [28]:
path = os.path.join(folder, 'X_test_scaled.csv')
X_test.to_csv(path, index=False)

We have checked that all of out Train and Test sets are encoded and scaled properly. 

In [29]:
X_train

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
9254,0.666667,0.817647,0.820702,0.315385,0.920588,0.296000,0.213000,0.129452,0.513889,0.937664,0.000000,0.25,0.666667,0.25,1.00,0.503420,1.0,0.556441,0.25
1561,1.000000,0.323529,0.763158,0.423077,0.167647,0.120667,0.286286,0.579256,0.311275,0.734416,0.000000,0.75,0.666667,0.50,0.75,0.922025,1.0,0.221356,1.00
1670,0.333333,0.347059,0.587368,0.007692,0.167647,0.211333,0.677857,0.643618,0.566993,0.394971,0.500000,1.00,1.000000,0.00,0.75,0.064295,1.0,0.178814,0.00
6087,0.666667,0.558824,0.870000,0.361538,0.041176,0.221000,0.451429,0.482633,0.205882,0.302252,0.666667,0.00,0.666667,0.50,0.25,0.451436,0.5,0.996441,0.75
6669,0.333333,0.020588,0.360175,0.992308,0.629412,0.870000,0.567000,0.834766,0.440359,0.332635,1.000000,0.75,1.000000,1.00,1.00,0.318057,0.5,0.027119,0.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,0.333333,0.364706,0.334912,0.800000,0.588235,0.940000,0.263571,0.929453,0.258987,0.938188,0.000000,0.25,0.333333,0.50,0.75,0.577975,0.5,0.075932,0.25
5191,1.000000,0.300000,0.608070,0.192308,0.176471,0.912000,0.976000,0.865515,0.331699,0.690938,0.666667,1.00,0.333333,0.25,0.25,0.668263,0.5,0.628305,0.00
5390,1.000000,0.885294,0.342632,0.669231,0.147059,0.921000,0.229286,0.326862,0.022059,0.686223,1.000000,0.00,0.666667,0.00,0.75,0.814637,1.0,0.205932,0.00
860,1.000000,0.455882,0.084561,0.453846,0.167647,0.501333,0.347429,0.409525,0.467320,0.321111,0.833333,0.75,0.666667,1.00,0.25,0.647059,1.0,0.042203,0.00


In [30]:
X_test

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
6252,0.000000,0.897059,0.127193,0.676923,0.385294,0.444000,0.248286,0.241103,0.480392,0.999476,0.833333,0.75,1.000000,0.25,0.25,0.486320,1.0,0.276780,0.75
4684,0.333333,0.552941,0.095965,0.576923,0.794118,0.173667,0.290143,0.768872,0.058007,0.541645,0.666667,0.00,0.333333,0.50,0.25,0.688782,1.0,0.373390,0.75
1731,0.000000,0.223529,0.052632,0.469231,0.088235,0.487333,0.655571,0.004257,0.321895,0.261393,1.000000,0.00,0.666667,1.00,0.75,0.116279,0.5,-0.000169,0.75
4742,1.000000,0.852941,0.536491,0.684615,0.423529,0.223667,0.707714,0.467901,0.084150,0.029859,0.666667,0.25,0.666667,0.50,0.25,0.603283,0.5,-0.000169,0.50
4521,0.000000,0.611765,0.685263,0.661538,0.688235,0.096667,0.619857,0.124918,0.285131,0.133054,0.500000,0.25,1.000000,1.00,0.25,0.685363,0.5,0.426102,0.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6412,0.000000,0.073529,0.732807,0.392308,0.850000,0.326667,0.420714,0.890414,0.090686,0.520691,0.666667,1.00,0.333333,0.25,0.25,0.994528,0.5,0.671017,0.00
8285,0.666667,0.955882,0.580526,0.869231,0.426471,0.183667,0.687857,0.281770,0.020425,0.037716,0.333333,1.00,1.000000,0.50,0.50,0.954856,1.0,-0.000169,0.00
7853,1.000000,0.526471,0.608596,0.946154,0.741176,0.133333,0.553571,0.116856,0.215686,0.475118,0.833333,1.00,0.333333,1.00,1.00,0.742134,0.5,0.986949,1.00
1095,0.333333,0.741176,0.979825,0.676923,0.955882,0.484000,0.813000,0.441778,0.164216,0.576742,0.666667,0.25,0.666667,0.25,0.50,0.390561,0.5,-0.000169,0.75


In [31]:
X_train.columns

Index(['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region'],
      dtype='object')

In [32]:
X_test_enc = pd.read_csv(r'C:\Irrigation_Water_Requirement\Data\Preprocesssed_Data\encoded_data\X_test_encoder.csv')
X_train_enc = pd.read_csv(r'C:\Irrigation_Water_Requirement\Data\Preprocesssed_Data\encoded_data\X_train_encoder.csv')

# Feature Engineering

    ~# As I am using tree based algorithms,we need to use encoded set for enginnering 

Added features

    ~Total_Water: Total_Water = Rainfall_mm + Previous_Irrigation_mm
    ~Water_per_Hectare = (Rainfall_mm + Previous_Irrigation_mm) / Field_Area_hectare
    ~Evaporation = Temperature_C × Sunlight_Hours × Wind_Speed_kmh
    ~Soil_Fertility = Organic_Carbon / (Electrical_Conductivity + 1)
    ~Mulch_Effect = Mulching_Used × Soil_Moisture
    ~Heat_Stress = Temperature_C × Humidity


In [33]:
# Total water recieved

X_train_enc['Total_Water'] = X_train_enc.Rainfall_mm + X_train_enc.Previous_Irrigation_mm
X_test_enc['Total_Water'] = X_test_enc.Rainfall_mm + X_test_enc.Previous_Irrigation_mm


In [34]:
# Evaporation
X_train_enc['Evaporation'] = X_train_enc.Temperature_C * X_train_enc.Sunlight_Hours * X_train_enc.Wind_Speed_kmh
X_test_enc['Evaporation'] = X_test_enc.Temperature_C * X_test_enc.Sunlight_Hours * X_test_enc.Wind_Speed_kmh

In [35]:
# Soil_Fertility
X_train_enc['Soil_Fertility'] = X_train_enc.Organic_Carbon / (X_train_enc.Electrical_Conductivity + 1)
X_test_enc['Soil_Fertility'] = X_test_enc.Organic_Carbon / (X_test_enc.Electrical_Conductivity + 1)

In [36]:
# Mulch_Effect

X_train_enc['Mulch_Effect'] = X_train_enc.Mulching_Used + X_train_enc.Soil_Moisture
X_test_enc['Mulch_Effect'] = X_test_enc.Mulching_Used + X_test_enc.Soil_Moisture

In [37]:
# Heat_Stress
X_train_enc['Heat_Stress'] = X_train_enc.Temperature_C * X_train_enc.Humidity
X_test_enc['Heat_Stress'] = X_test_enc.Temperature_C * X_test_enc.Humidity

In [38]:
X_train_enc.sample(20)

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Total_Water,Evaporation,Soil_Fertility,Mulch_Effect,Heat_Stress
4282,1,6.46,49.19,0.65,2.73,14.02,33.74,597.77,6.46,579,...,1,1116,2,3900,0,4497.77,52439.5668,0.174263,51.19,473.0348
3589,0,6.52,37.85,1.05,2.61,36.13,71.82,12.50,7.92,649,...,1,138,2,3999,1,4011.50,185711.0904,0.290859,39.85,2594.8566
6996,1,8.02,12.31,0.89,3.07,18.52,28.12,497.07,10.90,1505,...,2,812,2,2697,4,3194.07,303811.3400,0.218673,14.31,520.7824
7746,1,6.23,46.16,1.10,0.58,38.69,66.06,124.58,7.90,96,...,4,296,1,3395,3,3519.58,29342.4960,0.696203,47.16,2555.8614
4567,1,7.66,44.16,0.96,2.39,13.49,70.30,2331.69,4.31,968,...,3,730,1,5498,0,7829.69,56281.3592,0.283186,45.16,948.3470
2192,3,6.01,62.67,1.35,2.81,30.13,68.90,977.28,8.75,1310,...,2,1022,1,5257,4,6234.28,345365.1250,0.354331,63.67,2075.9570
1757,2,7.22,30.56,1.03,1.80,40.67,38.33,149.60,8.23,1501,...,3,1013,1,1454,0,1603.60,502405.8641,0.367857,31.56,1558.8811
5938,2,7.72,38.32,0.96,3.10,38.87,77.17,1643.74,4.18,766,...,2,113,1,3518,0,5161.74,124457.0756,0.234146,39.32,2999.5979
5367,1,4.96,35.16,0.98,1.42,12.53,44.44,378.00,4.22,864,...,1,1028,2,2955,2,3333.00,45685.3824,0.404959,37.16,556.8332
960,2,6.23,47.75,1.00,3.02,13.35,32.55,1087.28,4.57,454,...,2,868,2,3823,2,4910.28,27698.3130,0.248756,49.75,434.5425


Skewnes()

    If skewness in a column is larger then 0.80, we do log transformation

In [39]:
skewness = X_train_enc.skew()
skew_features = skewness[abs(skewness) > 0.60].index.to_list()

In [40]:
skew_features

['Evaporation', 'Soil_Fertility', 'Heat_Stress']

In [41]:
X_train_transformed = X_train_enc.copy()
X_test_transformed = X_test_enc.copy()


for col in skew_features:
    if (X_train_transformed[col]>= 0).all():
        X_train_transformed[col] = np.log1p(X_train_transformed[col])
        X_test_transformed[col] = np.log1p(X_test_transformed[col])

c:\Users\jyusu\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Feature Selection with Random Forest Classifier

In [42]:
rf_imp = RandomForestClassifier(
    n_jobs=-1, 
    random_state=42
)

rf_imp.fit(X_train_transformed, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [43]:
# Importance of features 

importance = rf_imp.feature_importances_


imp_df = pd.DataFrame({
    'feature': X_train_transformed.columns,
    'importance': importance
}).sort_values('importance', ascending=False)

imp_df['cumulative'] = imp_df['importance'].cumsum()

selected_features = imp_df.loc[
    imp_df['cumulative'] <= 0.90, 'feature'
]


In [44]:
selected_features

11         Crop_Growth_Stage
2              Soil_Moisture
7                Rainfall_mm
22              Mulch_Effect
9             Wind_Speed_kmh
5              Temperature_C
16             Mulching_Used
20               Evaporation
23               Heat_Stress
19               Total_Water
6                   Humidity
8             Sunlight_Hours
15        Field_Area_hectare
17    Previous_Irrigation_mm
Name: feature, dtype: object

In [45]:
X_train_sel = X_train_transformed.loc[:, selected_features]
X_test_sel = X_test_transformed.loc[:, selected_features]

In [46]:
X_train_sel

,Crop_Growth_Stage,Soil_Moisture,Rainfall_mm,Mulch_Effect,Wind_Speed_kmh,Temperature_C,Mulching_Used,Evaporation,Heat_Stress,Total_Water,Humidity,Sunlight_Hours,Field_Area_hectare,Previous_Irrigation_mm
0,1,54.78,323.92,56.78,1790,20.88,2,12.859938,6.726618,3606.92,39.91,10.29,736,3283
1,3,51.50,1448.12,53.50,1402,15.62,2,12.049618,6.557524,2754.12,45.04,7.81,1348,1306
2,4,41.48,1608.98,43.48,754,18.34,2,11.926909,7.192733,2663.98,72.45,10.94,94,1055
3,0,57.59,1206.63,58.59,577,18.63,1,11.157504,6.961730,7085.63,56.60,6.52,660,5879
4,3,28.53,2086.72,29.53,635,38.10,1,12.333489,7.810227,2246.72,64.69,9.39,465,160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,1,27.09,2323.37,28.09,1791,40.20,1,13.154304,7.466050,2771.37,43.45,7.17,845,448
7996,4,42.66,2163.57,43.66,1319,39.36,1,12.944295,8.209057,5870.57,93.32,8.06,977,3707
7997,0,27.53,817.31,29.53,1310,39.63,2,12.308987,7.394992,2032.31,41.05,4.27,1191,1215
7998,3,12.82,1023.91,14.82,613,27.04,2,11.989874,7.196396,1272.91,49.32,9.72,946,249


In [47]:
X_test

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
6252,0.000000,0.897059,0.127193,0.676923,0.385294,0.444000,0.248286,0.241103,0.480392,0.999476,0.833333,0.75,1.000000,0.25,0.25,0.486320,1.0,0.276780,0.75
4684,0.333333,0.552941,0.095965,0.576923,0.794118,0.173667,0.290143,0.768872,0.058007,0.541645,0.666667,0.00,0.333333,0.50,0.25,0.688782,1.0,0.373390,0.75
1731,0.000000,0.223529,0.052632,0.469231,0.088235,0.487333,0.655571,0.004257,0.321895,0.261393,1.000000,0.00,0.666667,1.00,0.75,0.116279,0.5,-0.000169,0.75
4742,1.000000,0.852941,0.536491,0.684615,0.423529,0.223667,0.707714,0.467901,0.084150,0.029859,0.666667,0.25,0.666667,0.50,0.25,0.603283,0.5,-0.000169,0.50
4521,0.000000,0.611765,0.685263,0.661538,0.688235,0.096667,0.619857,0.124918,0.285131,0.133054,0.500000,0.25,1.000000,1.00,0.25,0.685363,0.5,0.426102,0.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6412,0.000000,0.073529,0.732807,0.392308,0.850000,0.326667,0.420714,0.890414,0.090686,0.520691,0.666667,1.00,0.333333,0.25,0.25,0.994528,0.5,0.671017,0.00
8285,0.666667,0.955882,0.580526,0.869231,0.426471,0.183667,0.687857,0.281770,0.020425,0.037716,0.333333,1.00,1.000000,0.50,0.50,0.954856,1.0,-0.000169,0.00
7853,1.000000,0.526471,0.608596,0.946154,0.741176,0.133333,0.553571,0.116856,0.215686,0.475118,0.833333,1.00,0.333333,1.00,1.00,0.742134,0.5,0.986949,1.00
1095,0.333333,0.741176,0.979825,0.676923,0.955882,0.484000,0.813000,0.441778,0.164216,0.576742,0.666667,0.25,0.666667,0.25,0.50,0.390561,0.5,-0.000169,0.75


# Model Training

Random Forest Classifier

In [48]:
rf = RandomForestClassifier(
    n_jobs=-1,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train_sel, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [49]:
rf_pred = rf.predict(X_test_sel)

rf_accuracy_imp = accuracy_score(y_test, rf_pred)
rf_precision_imp = precision_score(y_test, rf_pred, average='weighted', zero_division=True)
rf_recall_imp = recall_score(y_test, rf_pred, zero_division= True, average='weighted')
rf_f1_imp= f1_score(y_test, rf_pred, average='weighted')

Decision Tree Classifier

In [50]:
dt = DecisionTreeClassifier()

dt.fit(X_train_sel, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [51]:
dt_pred_imp = dt.predict(X_test_sel)

dt_accuracy_imp = accuracy_score(y_test, dt_pred_imp)
dt_precision_imp = precision_score(y_test, dt_pred_imp, zero_division=True, average='weighted')
dt_recall_imp = recall_score(y_test, dt_pred_imp, average='weighted')
dt_f1_imp = f1_score(y_test, dt_pred_imp, average='weighted')

XGBoost Classifier

In [52]:
xgb_imp = XGBClassifier()

xgb_imp.fit(X_train_sel, y_train)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [53]:
xgb_pred_imp = xgb_imp.predict(X_test_sel)

xgb_accuracy_imp = accuracy_score(y_test, xgb_pred_imp)
xgb_precision_imp = precision_score(y_test, xgb_pred_imp, zero_division=True, average='weighted')
xgb_recall_imp = recall_score(y_test, xgb_pred_imp, average='weighted')
xgb_f1_imp = f1_score(y_test, xgb_pred_imp, average='weighted')

In [54]:
metrics = [
    ['Decision Tree Classifier', dt_accuracy_imp, dt_precision_imp, dt_recall_imp, dt_f1_imp],
    ['Random Forest Classifier', rf_accuracy_imp, rf_precision_imp, rf_recall_imp, rf_f1_imp],
    ['XGBoost', xgb_accuracy_imp, xgb_precision_imp, xgb_recall_imp, xgb_f1_imp]
]

headers = ['Accuracy Score', 'Precision Score', 'Recall Score', 'F1 Score']
print(tabulate(metrics, headers, floatfmt='.5f', tablefmt='orgtbl'))

|                          |   Accuracy Score |   Precision Score |   Recall Score |   F1 Score |
|--------------------------+------------------+-------------------+----------------+------------|
| Decision Tree Classifier |          0.98350 |           0.98335 |        0.98350 |    0.98341 |
| Random Forest Classifier |          0.97050 |           0.97153 |        0.97050 |    0.96864 |
| XGBoost                  |          0.99200 |           0.99201 |        0.99200 |    0.99198 |


# Hyperparametr Tuning

Grid Search CV for Decison Tree

In [55]:
param_grid = {
    # Tree depth 
    "max_depth": [None, 5, 10, 15, 20, 30],
    # Minimum samples to split a node
    "min_samples_split": [2, 5, 10, 20],
    # Minimum samples in a leaf
    "min_samples_leaf": [1, 2, 5, 10, 20],
    # Feature selection at each split
    "max_features": [None, "sqrt", "log2"],
    # Impurity measure
    "criterion": ["gini", "entropy", "log_loss"],
    # Handling class imbalance
    "class_weight": [
        None,
        "balanced",
    ]
}


gs_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1
)

gs_dt.fit(X_train_sel, y_train)

best_dt = gs_dt.best_estimator_

In [56]:
best_dt

,criterion,'entropy'
,splitter,'best'
,max_depth,None
,min_samples_split,5
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [57]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    # Forest size
    "n_estimators": [100, 200, 300],
    # Tree complexity
    "max_depth": [None, 10, 20, 30],
    "min_samples_leaf": [1, 2, 5, 10],
    # Class imbalance handling
    "criterion": ["gini", "entropy"]
}

gs_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_grid,
    cv=5, 
    n_jobs=-1
)
gs_rf.fit(X_train_sel, y_train)

best_rf = gs_rf.best_estimator_

In [58]:
print(best_rf)

RandomForestClassifier(criterion='entropy', max_depth=30, n_estimators=200,
                       n_jobs=-1, random_state=42)


XGBoost Hyperparametr Tuning

In [59]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    # Boosting rounds
    "n_estimators": [100, 200, 300, 400],

    # Tree depth & structure
    "max_depth": [3, 5, 7, 10],
    "min_child_weight": [1, 3, 5],

    # Learning rate
    "learning_rate": [0.01, 0.05, 0.1],

    # Regularization
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.1, 0.5],
    "reg_lambda": [1, 1.5, 2],
}


gs_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=42),
    param_distributions=param_grid,
    cv=5, 
    n_jobs=-1
)

gs_xgb.fit(X_train_sel, y_train)

best_xgb = gs_xgb.best_estimator_

In [60]:
best_xgb

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [61]:
dt_pred_hyper = gs_dt.predict(X_test_sel)

dt_accuracy_imp_2 = accuracy_score(y_test, dt_pred_hyper)
dt_precision_imp_2 = precision_score(y_test, dt_pred_hyper, zero_division=True, average='weighted')
dt_recall_imp_2 = recall_score(y_test, dt_pred_hyper, average='weighted')
dt_f1_imp_2 = f1_score(y_test, dt_pred_hyper, average='weighted')

In [62]:
rf_pred_hyper = gs_rf.predict(X_test_sel)

rf_accuracy_imp_2 = accuracy_score(y_test, rf_pred_hyper)
rf_precision_imp_2 = precision_score(y_test, rf_pred_hyper, zero_division=True, average='weighted')
rf_recall_imp_2 = recall_score(y_test, rf_pred_hyper, zero_division= True, average='weighted')
rf_f1_imp_2 = f1_score(y_test, rf_pred_hyper, average='weighted')

In [63]:
xgb_pred_hyper = gs_xgb.predict(X_test_sel)

xgb_accuracy_imp_2 = accuracy_score(y_test, xgb_pred_hyper)
xgb_precision_imp_2 = precision_score(y_test, xgb_pred_hyper, zero_division=True, average='weighted')
xgb_recall_imp_2 = recall_score(y_test, xgb_pred_hyper, average='weighted')
xgb_f1_imp_2 = f1_score(y_test, xgb_pred_hyper, average='weighted')

In [64]:
metrics = [
    ['Decision Tree Classifier', dt_accuracy_imp, dt_precision_imp, dt_recall_imp, dt_f1_imp],
    ['Random Forest Classifier', rf_accuracy_imp, rf_precision_imp, rf_recall_imp, rf_f1_imp],
    ['XGBoost', xgb_accuracy_imp, xgb_precision_imp, xgb_recall_imp, xgb_f1_imp]
]

headers = ['Accuracy Score', 'Precision Score', 'Recall Score', 'F1 Score']
print(tabulate(metrics, headers, floatfmt='.5f', tablefmt='orgtbl'))

print('\n==========Hyperparametr Tuning===========')
metrics = [
    ['Decision Tree Classifier', dt_accuracy_imp_2, dt_precision_imp_2, dt_recall_imp_2, dt_f1_imp_2],
    ['Random Forest Classifier', rf_accuracy_imp_2, rf_precision_imp_2, rf_recall_imp_2, rf_f1_imp_2],
    ['XGBoost', xgb_accuracy_imp_2, xgb_precision_imp_2, xgb_recall_imp_2, xgb_f1_imp_2]
]

headers = ['Accuracy Score', 'Precision Score', 'Recall Score', 'F1 Score']
print(tabulate(metrics, headers, floatfmt='.5f', tablefmt='orgtbl'))

|                          |   Accuracy Score |   Precision Score |   Recall Score |   F1 Score |
|--------------------------+------------------+-------------------+----------------+------------|
| Decision Tree Classifier |          0.98350 |           0.98335 |        0.98350 |    0.98341 |
| Random Forest Classifier |          0.97050 |           0.97153 |        0.97050 |    0.96864 |
| XGBoost                  |          0.99200 |           0.99201 |        0.99200 |    0.99198 |

==========Hyperparametr Tuning===========
|                          |   Accuracy Score |   Precision Score |   Recall Score |   F1 Score |
|--------------------------+------------------+-------------------+----------------+------------|
| Decision Tree Classifier |          0.98900 |           0.98894 |        0.98900 |    0.98894 |
| Random Forest Classifier |          0.97950 |           0.97968 |        0.97950 |    0.97931 |
| XGBoost                  |          0.98900 |           0.98894 |        

In [65]:
from pathlib import Path
import joblib

MODELS_DIR = Path("C:/Irrigation_Water_Requirement/Models/Best_model")
model_path = MODELS_DIR / "XGBoost_model.pkl"
joblib.dump(gs_xgb, model_path)

print(f"Model saved at: {model_path}")

Model saved at: C:\Irrigation_Water_Requirement\Models\Best_model\XGBoost_model.pkl


In [66]:
MODELS_DIR = Path("C:/Irrigation_Water_Requirement/Models/Other_models")
model_path = MODELS_DIR / "Random_Forest.pkl"
joblib.dump(gs_rf, model_path)

print(f"Model saved at: {model_path}")

Model saved at: C:\Irrigation_Water_Requirement\Models\Other_models\Random_Forest.pkl


In [67]:
MODELS_DIR = Path("C:/Irrigation_Water_Requirement/Models/Other_models")
model_path = MODELS_DIR / "Decision_tree.pkl"
joblib.dump(gs_dt, model_path)

print(f"Model saved at: {model_path}")

Model saved at: C:\Irrigation_Water_Requirement\Models\Other_models\Decision_tree.pkl


# XAI With SHAP

In [68]:
import shap
import matplotlib_inline as plt

In [ ]:
X_shap = X_test_enc.sample(60, random_state=42)

: 

In [ ]:
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_shap)